[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/PacktPublishing/Data-Strategy-for-LLMs/blob/main/chapter_08/Jupyter_Notebooks/Chapter_8_Notebook.ipynb)

**Click the badge above to run this notebook in Google Colab (no local setup needed).**

# Chapter 8: LLM Model Customization

This notebook contains practical examples for customizing domain-specific LLM assistants at the system level, using the data assets and customization techniques introduced throughout the book.

## 1. Set Up the Chapter Environment



1. Run the book-wide setup once from the repository root: `bash setup/setup_mac.sh` (macOS/Linux) or `powershell -ExecutionPolicy Bypass -File setup/setup_windows.ps1` (Windows).

2. Select the **"Python (Data Strategy Book)"** kernel. If it is missing, reload the VS Code window.



This setup cell locates the repository in local and Colab environments, installs only missing tabular and similarity packages, and creates the Chapter 8 dataset directory.

In [1]:
import importlib.util
import json
import re
import subprocess
import sys
from datetime import datetime, timezone
from hashlib import sha256
from pathlib import Path


def install_if_missing(module_name, package_name=None):
    if importlib.util.find_spec(module_name) is None:
        package_name = package_name or module_name
        subprocess.run(
            [sys.executable, "-m", "pip", "install", package_name, "--quiet"],
            check=True,
        )


install_if_missing("pandas")
install_if_missing("sklearn", "scikit-learn")

import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

repo_root = Path.cwd()
for candidate in [Path.cwd(), *Path.cwd().parents]:
    if (candidate / "setup" / "requirements.txt").exists():
        repo_root = candidate
        break

chapter_dir = repo_root / "chapter_08"
dataset_dir = chapter_dir / "datasets"
dataset_dir.mkdir(parents=True, exist_ok=True)

document_path = dataset_dir / "knowledge_documents.jsonl"
structured_data_path = dataset_dir / "employee_vacation_balances.csv"
registry_path = dataset_dir / "source_of_truth_registry.csv"
conversation_path = dataset_dir / "behavioral_conversations.jsonl"

with document_path.open("r", encoding="utf-8") as file:
    knowledge_documents = [json.loads(line) for line in file if line.strip()]
vacation_balances = pd.read_csv(structured_data_path)
source_registry = pd.read_csv(registry_path)
with conversation_path.open("r", encoding="utf-8") as file:
    historical_conversations = [json.loads(line) for line in file if line.strip()]

pd.set_option("display.max_colwidth", 90)
print(f"Repository root: {repo_root}")
print(f"Documents: {document_path} ({len(knowledge_documents)} records)")
print(f"Structured data: {structured_data_path} ({len(vacation_balances)} records)")
print(f"Source registry: {registry_path}")
print(f"Conversations: {conversation_path} ({len(historical_conversations)} records)")
print("Setup complete.")

Repository root: d:\Code\Data-Strategy-for-LLMs
Documents: d:\Code\Data-Strategy-for-LLMs\chapter_08\datasets\knowledge_documents.jsonl (8 records)
Structured data: d:\Code\Data-Strategy-for-LLMs\chapter_08\datasets\employee_vacation_balances.csv (2 records)
Source registry: d:\Code\Data-Strategy-for-LLMs\chapter_08\datasets\source_of_truth_registry.csv
Conversations: d:\Code\Data-Strategy-for-LLMs\chapter_08\datasets\behavioral_conversations.jsonl (5 records)
Setup complete.


## Preparing Knowledge Data for Customization

The cell below loads the shared HR data, selects authoritative records, exports review artifacts, and verifies the result.

### Cases Covered

| Case | Example | Expected handling |
|---|---|---|
| Current authoritative policy | Approved travel policy from the HR Portal | Include |
| Exact duplicate | Identical policy copied to a shared drive | Exclude as a duplicate |
| Outdated near duplicate | Older approved policy in onboarding material | Exclude and flag conflicting values |
| Newer draft | Unapproved policy with changed limits | Exclude and flag for review |
| Local guidance | Team wiki copy with different requirements | Exclude as non-authoritative and flag for review |
| Superseded authoritative version | Older vacation policy from the HR Portal | Replace with the latest approved version |
| Structured system facts | Current employee vacation balances from HRIS | Include and route employee-specific questions here |
| Invalid record | Document with missing content and malformed metadata | Quarantine with validation reasons |

In [2]:
# Convert structured rows to the common knowledge-record schema.
structured_records = []
for row in vacation_balances.to_dict("records"):
    as_of_date = str(row["as_of_date"])
    employee_id = row["employee_id"]
    structured_records.append({
        "document_id": f"HRIS-BAL-{employee_id}",
        "policy_id": f"VACATION_BALANCE:{employee_id}",
        "knowledge_type": "vacation_balance",
        "title": f"Vacation balance for {employee_id}",
        "version": as_of_date.replace("-", "."),
        "status": row["status"],
        "owner": row["owner"],
        "source_system": row["source_system"],
        "source_type": "structured_record",
        "effective_date": as_of_date,
        "last_reviewed": as_of_date,
        "content": f"Employee {employee_id} has {row['vacation_balance_days']:g} vacation days remaining as of {as_of_date}.",
    })

raw_records = [*knowledge_documents, *structured_records]

# Validate content and metadata; invalid records go to quarantine.
required_fields = [
    "document_id", "policy_id", "knowledge_type", "title", "version", "status",
    "owner", "source_system", "source_type", "effective_date", "last_reviewed", "content",
]
valid_statuses = {"approved", "published", "draft", "current"}


def parse_version(value):
    match = re.match(r"^(\d+)(?:\.(\d+))?", str(value).strip())
    return (int(match.group(1)), int(match.group(2) or 0)) if match else None


def validate_record(record):
    normalized = dict(record)
    reasons = [
        f"missing {field}" for field in required_fields
        if field not in normalized or pd.isna(normalized[field]) or not str(normalized[field]).strip()
    ]
    normalized["status"] = str(normalized.get("status", "")).strip().lower()
    if normalized["status"] not in valid_statuses:
        reasons.append("unrecognized status")
    normalized["version_key"] = parse_version(normalized.get("version"))
    if normalized["version_key"] is None:
        reasons.append("invalid version")
    for field in ("effective_date", "last_reviewed"):
        normalized[f"{field}_parsed"] = pd.to_datetime(normalized.get(field), errors="coerce")
        if pd.isna(normalized[f"{field}_parsed"]):
            reasons.append(f"invalid {field}")
    return normalized, sorted(set(reasons))


valid_records, quarantined_records = [], []
for record in raw_records:
    normalized, reasons = validate_record(record)
    if reasons:
        quarantined_records.append({**record, "validation_reasons": "; ".join(reasons)})
    else:
        valid_records.append(normalized)
valid_df = pd.DataFrame(valid_records)
quarantine_df = pd.DataFrame(quarantined_records)

# Group related versions and detect exact and near duplicates.
version_history = valid_df.sort_values(
    ["policy_id", "effective_date_parsed", "version_key"]
)[["policy_id", "document_id", "version", "status", "source_system", "effective_date"]]


def normalize_text(text):
    return re.sub(r"\s+", " ", re.sub(r"[^a-z0-9$.,]", " ", text.lower())).strip()


valid_df["normalized_content"] = valid_df["content"].map(normalize_text)
valid_df["content_hash"] = valid_df["normalized_content"].map(
    lambda text: sha256(text.encode("utf-8")).hexdigest()
)
exact_duplicates = (
    valid_df.groupby("content_hash")["document_id"].agg(list)
    .loc[lambda groups: groups.map(len) > 1]
)

near_duplicate_rows = []
for policy_id, group in valid_df.groupby("policy_id"):
    if len(group) < 2:
        continue
    group = group.reset_index(drop=True)
    matrix = TfidfVectorizer(ngram_range=(1, 2)).fit_transform(group["normalized_content"])
    similarities = cosine_similarity(matrix)
    for left_index in range(len(group)):
        for right_index in range(left_index + 1, len(group)):
            score = float(similarities[left_index, right_index])
            if score >= 0.45:
                near_duplicate_rows.append({
                    "policy_id": policy_id,
                    "document_a": group.loc[left_index, "document_id"],
                    "document_b": group.loc[right_index, "document_id"],
                    "similarity": round(score, 3),
                    "exact_duplicate": group.loc[left_index, "content_hash"] == group.loc[right_index, "content_hash"],
                })
near_duplicates_df = pd.DataFrame(near_duplicate_rows)

# Keep the latest approved/current record from each designated source of truth.
registry_lookup = source_registry.set_index("knowledge_type").to_dict("index")
selection_rows, selected_ids = [], set()
for policy_id, group in valid_df.groupby("policy_id"):
    knowledge_type = group["knowledge_type"].iloc[0]
    authority = registry_lookup[knowledge_type]
    eligible_statuses = {"current"} if group["source_type"].eq("structured_record").all() else {"approved"}
    eligible = group[
        group["status"].isin(eligible_statuses)
        & group["source_system"].eq(authority["source_system"])
        & group["owner"].eq(authority["owner"])
    ]
    if eligible.empty:
        raise ValueError(f"No authoritative record for {policy_id}")
    selected = eligible.sort_values(
        ["effective_date_parsed", "version_key"], ascending=False
    ).iloc[0]
    selected_ids.add(selected["document_id"])
    for _, record in group.iterrows():
        if record["document_id"] == selected["document_id"]:
            decision, reason = "include", "latest authoritative record"
        elif record["content_hash"] == selected["content_hash"]:
            decision, reason = "exclude", "exact duplicate of selected record"
        elif record["source_system"] != authority["source_system"] or record["owner"] != authority["owner"]:
            decision, reason = "exclude", "not from designated source of truth"
        elif record["status"] not in eligible_statuses:
            decision, reason = "exclude", f"status is {record['status']}"
        else:
            decision, reason = "exclude", "superseded authoritative version"
        selection_rows.append({
            "document_id": record["document_id"],
            "decision": decision,
            "selection_reason": reason,
        })
selection_df = pd.DataFrame(selection_rows)
selected_df = valid_df[valid_df["document_id"].isin(selected_ids)].copy()

# Flag similar documents whose important values differ from the selected version.
def important_values(text):
    values = re.findall(r"\$[\d,]+|\b\d+(?:\.\d+)?\b|manager|director|vice president", text.lower())
    return sorted(set(values))


review_rows = []
for _, candidate in valid_df[~valid_df["document_id"].isin(selected_ids)].iterrows():
    selected_matches = selected_df[selected_df["policy_id"].eq(candidate["policy_id"])]
    if selected_matches.empty or candidate["content_hash"] in set(selected_matches["content_hash"]):
        continue
    selected = selected_matches.iloc[0]
    pair_text = [selected["normalized_content"], candidate["normalized_content"]]
    similarity = float(
        cosine_similarity(TfidfVectorizer(ngram_range=(1, 2)).fit_transform(pair_text))[0, 1]
    )
    selected_values = important_values(selected["content"])
    candidate_values = important_values(candidate["content"])
    if similarity >= 0.45 and selected_values != candidate_values:
        review_rows.append({
            "policy_id": candidate["policy_id"],
            "selected_document_id": selected["document_id"],
            "candidate_document_id": candidate["document_id"],
            "candidate_source": candidate["source_system"],
            "similarity": round(similarity, 3),
            "selected_values": ", ".join(selected_values),
            "candidate_values": ", ".join(candidate_values),
            "review_reason": "similar text contains different limits or approval requirements",
            "candidate_passage": candidate["content"],
        })
review_df = pd.DataFrame(review_rows)

# Route employee facts to structured data and explanatory questions to policy documents.
def route_question(question):
    normalized = question.lower()
    if any(phrase in normalized for phrase in ("how many", "left", "remaining", "my balance")):
        knowledge_type = "vacation_balance"
    elif any(phrase in normalized for phrase in ("how is", "calculated", "accrue", "carry over")):
        knowledge_type = "vacation_calculation"
    else:
        knowledge_type = "travel_policy"
    authority = registry_lookup[knowledge_type]
    return {
        "question": question,
        "knowledge_type": knowledge_type,
        "source_system": authority["source_system"],
    }


routing_df = pd.DataFrame([
    route_question("How many vacation days do I have left?"),
    route_question("How is my vacation balance calculated?"),
])

# Export cleaned, rejected, quarantined, and conflicting records with a manifest.
clean_path = dataset_dir / "cleaned_knowledge_collection.jsonl"
rejected_path = dataset_dir / "rejected_knowledge_collection.csv"
review_path = dataset_dir / "knowledge_review_queue.csv"
manifest_path = dataset_dir / "knowledge_manifest.json"
quarantine_path = dataset_dir / "quarantined_knowledge.csv"

export_columns = [
    "document_id", "policy_id", "knowledge_type", "title", "version", "status",
    "owner", "source_system", "source_type", "effective_date", "last_reviewed", "content", "content_hash",
]
with clean_path.open("w", encoding="utf-8") as file:
    for record in selected_df[export_columns].to_dict("records"):
        file.write(json.dumps(record) + "\n")
rejected_df = valid_df.merge(selection_df, on="document_id").query("decision == 'exclude'")
rejected_df[[*export_columns, "selection_reason"]].to_csv(rejected_path, index=False)
quarantine_df.to_csv(quarantine_path, index=False)
review_df.to_csv(review_path, index=False)

manifest = {
    "created_at": datetime.now(timezone.utc).isoformat(),
    "input_files": [document_path.name, structured_data_path.name, registry_path.name],
    "raw_count": len(raw_records),
    "valid_count": len(valid_df),
    "quarantined_count": len(quarantine_df),
    "index_ready_count": len(selected_df),
    "rejected_count": len(rejected_df),
    "manual_review_count": len(review_df),
    "source_systems": sorted(selected_df["source_system"].unique().tolist()),
    "selected_documents": sorted(selected_ids),
}
manifest_path.write_text(json.dumps(manifest, indent=2), encoding="utf-8")

# Reload and verify the exported data.
with clean_path.open("r", encoding="utf-8") as file:
    exported_records = [json.loads(line) for line in file if line.strip()]
exported_df = pd.DataFrame(exported_records)
exported_manifest = json.loads(manifest_path.read_text(encoding="utf-8"))
assert set(export_columns).issubset(exported_df.columns)
assert not exported_df["content_hash"].duplicated().any()
assert exported_df.query("source_type == 'policy'")["status"].eq("approved").all()
assert exported_df.query("source_type == 'structured_record'")["status"].eq("current").all()
for _, record in exported_df.iterrows():
    authority = registry_lookup[record["knowledge_type"]]
    assert record["source_system"] == authority["source_system"]
    assert record["owner"] == authority["owner"]
assert set(review_df["candidate_document_id"]).isdisjoint(exported_df["document_id"])
assert set(routing_df["source_system"]) == {"HRIS", "HR Portal"}
assert exported_manifest["index_ready_count"] == len(exported_df)
assert exported_manifest["raw_count"] == len(valid_df) + len(quarantine_df)

print(f"Raw: {len(raw_records)} | Valid: {len(valid_df)} | Quarantined: {len(quarantine_df)}")
print(f"Index-ready: {len(exported_df)} | Rejected: {len(rejected_df)} | Review: {len(review_df)}")
print("Exact duplicate groups:", exact_duplicates.to_dict())
print("\nSelected records:")
display(exported_df[["document_id", "knowledge_type", "source_system", "status"]])
print("Routing examples:")
display(routing_df)
print(f"\nQuality checks passed. Artifacts written to {dataset_dir}")

Raw: 10 | Valid: 9 | Quarantined: 1
Index-ready: 4 | Rejected: 5 | Review: 4
Exact duplicate groups: {'6e937033bebe7b14a3670e259e653313810bb4c90353ca9ca3cdc1f550fec2e5': ['TRV-PORTAL-V3', 'TRV-PORTAL-COPY']}

Selected records:


,document_id,knowledge_type,source_system,status
0,TRV-PORTAL-V3,travel_policy,HR Portal,approved
1,VAC-PORTAL-V2,vacation_calculation,HR Portal,approved
2,HRIS-BAL-E001,vacation_balance,HRIS,current
3,HRIS-BAL-E002,vacation_balance,HRIS,current


Routing examples:


,question,knowledge_type,source_system
0,How many vacation days do I have left?,vacation_balance,HRIS
1,How is my vacation balance calculated?,vacation_calculation,HR Portal



Quality checks passed. Artifacts written to d:\Code\Data-Strategy-for-LLMs\chapter_08\datasets


## Extracting High-Quality Behavioral Examples

The cell below removes non-task interactions, segments conversations by objective, keeps complete successful workflows, and exports examples for prompting and training.

### Cases Covered

| Case | Example | Expected handling |
|---|---|---|
| Multi-task conversation | Travel booking followed by a vacation-balance question | Split into two examples |
| Social and unrelated messages | Greetings, weather discussion, and thanks | Remove without discarding the task |
| Complete tool-assisted workflow | Clarification, policy lookup, tool result, and final response | Include the full task trace |
| Resolved but incomplete workflow | Corrected invoice with no captured intermediate actions | Exclude because steps are missing |
| Unsuccessful interaction | Laptop issue ending in escalation | Exclude because the outcome is not successful |
| Expert demonstration | Missing-receipt exception handled from policy | Include as a high-quality example |

In [2]:
# Extract task events instead of filtering entire conversations.
behavioral_examples = []
rejected_tasks = []
noise_event_count = 0

for conversation in historical_conversations:
    task_events = {}
    for event in sorted(conversation["events"], key=lambda item: item["sequence"]):
        if event.get("contribution") != "task" or not event.get("task_id"):
            noise_event_count += 1
            continue
        task_events.setdefault(event["task_id"], []).append(event)

    # Each task identifier marks an objective boundary, even when several tasks
    # appear in the same source conversation.
    for task_id, events in task_events.items():
        outcome = conversation["task_outcomes"].get(task_id, {})
        rejection_reasons = []
        if outcome.get("status") != "resolved":
            rejection_reasons.append("outcome is not successful")
        if not outcome.get("captured_steps_complete", False):
            rejection_reasons.append("intermediate steps are incomplete")

        user_messages = [
            event for event in events
            if event["role"] == "user" and event["event_type"] == "message"
        ]
        assistant_messages = [
            event for event in events
            if event["role"] == "assistant" and event["event_type"] == "message"
        ]
        if not user_messages:
            rejection_reasons.append("objective is missing")
        if not assistant_messages:
            rejection_reasons.append("final outcome is missing")

        if rejection_reasons:
            rejected_tasks.append({
                "conversation_id": conversation["conversation_id"],
                "task_id": task_id,
                "reasons": "; ".join(rejection_reasons),
            })
            continue

        behavioral_examples.append({
            "example_id": f"{conversation['conversation_id']}:{task_id}",
            "source_conversation_id": conversation["conversation_id"],
            "channel": conversation["channel"],
            "objective": user_messages[0]["content"],
            "outcome": assistant_messages[-1]["content"],
            "events": [
                {
                    key: event[key]
                    for key in ("role", "event_type", "content", "tool_name")
                    if key in event
                }
                for event in events
            ],
        })

# Few-shot examples retain the complete interaction, including tool use.
few_shot_examples = [
    {
        "example_id": example["example_id"],
        "messages": [
            {
                "role": event["role"],
                "content": event["content"],
                **({"tool_name": event["tool_name"]} if "tool_name" in event else {}),
            }
            for event in example["events"]
        ],
    }
    for example in behavioral_examples
]

# Instruction-response pairs provide a compact format for supervised training.
instruction_response_pairs = [
    {
        "example_id": example["example_id"],
        "instruction": example["objective"],
        "response": example["outcome"],
    }
    for example in behavioral_examples
]

few_shot_path = dataset_dir / "behavioral_few_shot_examples.jsonl"
instruction_path = dataset_dir / "behavioral_instruction_response_pairs.jsonl"
rejected_tasks_path = dataset_dir / "behavioral_rejected_tasks.csv"
behavioral_manifest_path = dataset_dir / "behavioral_manifest.json"

for path, records in (
    (few_shot_path, few_shot_examples),
    (instruction_path, instruction_response_pairs),
):
    with path.open("w", encoding="utf-8") as file:
        for record in records:
            file.write(json.dumps(record) + "\n")
pd.DataFrame(rejected_tasks).to_csv(rejected_tasks_path, index=False)

behavioral_manifest = {
    "created_at": datetime.now(timezone.utc).isoformat(),
    "source_conversations": len(historical_conversations),
    "task_interactions_found": len(behavioral_examples) + len(rejected_tasks),
    "examples_exported": len(behavioral_examples),
    "tasks_rejected": len(rejected_tasks),
    "non_task_events_removed": noise_event_count,
}
behavioral_manifest_path.write_text(
    json.dumps(behavioral_manifest, indent=2), encoding="utf-8"
)

# Verify segmentation, quality filters, and both export formats.
assert len(behavioral_examples) == 4
assert len(rejected_tasks) == 2
assert sum(
    example["source_conversation_id"] == "CONV-001"
    for example in behavioral_examples
) == 2
assert all(example["objective"] and example["outcome"] for example in behavioral_examples)
assert all(
    event["event_type"] in {"message", "tool_call", "tool_result"}
    for example in behavioral_examples
    for event in example["events"]
)
assert {item["task_id"] for item in rejected_tasks} == {
    "invoice_correction", "laptop_failure"
}
assert len(few_shot_examples) == len(instruction_response_pairs)

print(
    f"Conversations: {len(historical_conversations)} | "
    f"Tasks found: {len(behavioral_examples) + len(rejected_tasks)} | "
    f"Examples exported: {len(behavioral_examples)} | "
    f"Rejected: {len(rejected_tasks)}"
)
print(f"Non-task events removed: {noise_event_count}")
display(pd.DataFrame(behavioral_examples)[
    ["example_id", "channel", "objective", "outcome"]
])
print(f"\nBehavioral datasets written to {dataset_dir}")

Conversations: 5 | Tasks found: 6 | Examples exported: 4 | Rejected: 2
Non-task events removed: 8


,example_id,channel,objective,outcome
0,CONV-001:travel_booking,support_chat,Please book my trip to Berlin next month.,Your economy itinerary is booked for May 12-15. Confirmation: BER-481.
1,CONV-001:vacation_balance,support_chat,"Also, how many vacation days do I have left?",You have 8.5 vacation days remaining.
2,CONV-002:password_reset,support_ticket,I cannot sign in after changing phones.,Use the temporary registration pass sent to your corporate email to register the new p...
3,CONV-005:expense_exception,expert_demonstration,My hotel receipt is missing. Can I still submit the expense?,Submit a missing-receipt declaration with the $180 expense and attach the hotel's conf...



Behavioral datasets written to d:\Code\Data-Strategy-for-LLMs\chapter_08\datasets
